In [0]:
dbutils.widgets.text("catalog", "vp_rm20040829")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import *
from datetime import datetime

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ops.reconciliation_results (
    run_ts        TIMESTAMP,
    check_name    STRING,
    source_value  DECIMAL(18,3),
    target_value  DECIMAL(18,3),
    status        STRING
)
""")

run_ts = datetime.now()

source_sc = (
    spark.read.table(f"{catalog}.bronze.sessions")
    .select(countDistinct("session_id").alias("v"))
    .collect()[0]["v"]
)
target_sc = (
    spark.read.table(f"{catalog}.silver.sessions").count()
    + spark.read.table(f"{catalog}.silver.quarantine_sessions").count()
)
status_sc = "PASS" if source_sc == target_sc else "FAIL"

source_ek = (
    spark.read.table(f"{catalog}.silver.sessions")
    .filter(col("status") == "COMPLETED")
    .agg(coalesce(sum("energy_kwh"), lit(0)).alias("v"))
    .collect()[0]["v"]
)
target_ek = (
    spark.read.table(f"{catalog}.gold.agg_daily_station")
    .agg(coalesce(sum("energy_kwh"), lit(0)).alias("v"))
    .collect()[0]["v"]
)
status_ek = "PASS" if -0.001 < float(source_ek) - float(target_ek) < 0.001 else "FAIL"

source_cm = (
    spark.read.table(f"{catalog}.silver.members")
    .select(countDistinct("member_id").alias("v"))
    .collect()[0]["v"]
)
target_cm = (
    spark.read.table(f"{catalog}.gold.dim_members")
    .filter(col("is_current") == lit(True))
    .count()
)
status_cm = "PASS" if source_cm == target_cm else "FAIL"

results = (
    spark.createDataFrame(
        [
            (run_ts, "SESSION_COUNT",   float(source_sc), float(target_sc), status_sc),
            (run_ts, "ENERGY_KWH",      float(source_ek), float(target_ek), status_ek),
            (run_ts, "CURRENT_MEMBERS", float(source_cm), float(target_cm), status_cm),
        ],
        ["run_ts", "check_name", "source_value", "target_value", "status"],
    )
    .select(
        col("run_ts").cast("timestamp"),
        col("check_name"),
        col("source_value").cast("decimal(18,3)"),
        col("target_value").cast("decimal(18,3)"),
        col("status"),
    )
)

(results.write
    .mode("append")
    .saveAsTable(f"{catalog}.ops.reconciliation_results"))



In [0]:
%sql
select * from vp_rm20040829.ops.reconciliation_results